# Quickstart

In [1]:
import time
import numpy as np
import pandas as pd
import mlx.core as mx
from rdkit import Chem
from mlx_cluster import fp_to_mlx, get_tanimoto, butina, KMeans, FPGenerator

Helper Function

In [2]:
def total_time(total_time):
    total = float(round(total_time, 2))
    print(f"Total Time: {total} Sec")

# Data Prep
Dataset pulled from [BitBirch](https://github.com/mqcomplab/bblean.git)

In [3]:
# load molecules
df = pd.read_csv("dataset/chembl-33-natural-products-subset.smi", sep='\t', names=['smiles'], header=None)

# take first 10_000 molecules
smi_list = df['smiles'][:10_000].tolist()

print(f"The number of molecules for clustering: {len(smi_list)}")

The number of molecules for clustering: 10000


# Molecular Fingerprint
Fingerprints can be generated as usual using RDKit. However, this the modern RDKit fingerprint generation method has been wrapped into the **Generate()** class. This class can speed up fingerprint generation by using multiple cpus. The smiles strings needs to first be converted into a list or np.array of RDMols before proceeding. Converting the smi_list into a list/array of rdkit molecular objects can also be used. Here we will keep things simple and use a list/array of smiles strings.

In [4]:
# can be a list or np.array of smiles strings or ROMols.
fp_gen = FPGenerator(smi_list)
rdkit_fps = fp_gen.fingerprint(type='rdkit', nbits=1024, n_cpu=10)

Afterwards, the fingerprints must be converted into type mx.array(). This can be done in two lines or by using the support method **fp_to_mlx()**.

In [5]:
np_array = np.array(rdkit_fps)
mlx_fp = mx.array(np_array).astype(mx.float32)

# # optional support function to convert fp list into mx.array()
# mlx_fp = fp_to_mlx(rdkit_fps)

# Clustering

Currently, two clustering algorithms are available - Butina and KMeans.

## Generating Tanimoto Similarity Score
For Butina, this requires generating Tanimoto similarity scores between a list of molecules. This is handled internally in the **butina()** method. However, as Tanimoto score is calculated on mx.arrays, this can also be done quickly on the apple silicon as follows:

In [6]:
score = get_tanimoto(mlx_fp, output='array')
score

array([0.6756757 , 0.93318486, 0.9736842 , ..., 0.37985438, 0.3532338 ,
       0.10736579], shape=(49995000,), dtype=float32)

## Butina Clustering
The butina clustering is done by using the **butina()** method and giving it the mlx fingerprint array. The output will give the output as a list of clusters, with the first entry of a cluster being the centroid.

In [7]:
# record time
start = time.time()

butina_mlx = butina(mlx_fp)

# end time
end = time.time()
total_time(end - start)

Total Time: 0.38 Sec


In [8]:
print(f"Number of clusters: {len(butina_mlx)}")

Number of clusters: 2525


## KMeans
The Kmeans clustering should mimic what is seen with Scikit-Learn. The number of clusters are specified and then generated. The clusters and the centroids can also be extracted using support methods, as demoed below.

In [9]:
# record time
start = time.time()

# define and fit the model
kmeans = KMeans(n_clusters=200, random_state=0).fit(mlx_fp)

# end time
end = time.time()
total_time(end-start)

Total Time: 0.22 Sec


Extracting cluster and centroids.

In [10]:
# cluster labels
kcluster = np.array(kmeans.labels_)
kcluster

array([ 71,  71,  38, ..., 127, 103, 103], shape=(10000,), dtype=uint32)

In [11]:
# cluster centroid
kcentroid, _ = np.array(kmeans.pairwise_distances_argmin_min(mlx_fp))
kcentroid

array([4.300e+03, 1.234e+03, 6.922e+03, 3.760e+02, 6.918e+03, 2.807e+03,
       3.438e+03, 3.381e+03, 4.937e+03, 8.260e+02, 4.460e+03, 7.980e+03,
       5.688e+03, 9.913e+03, 2.706e+03, 9.370e+02, 5.329e+03, 4.561e+03,
       3.335e+03, 9.692e+03, 1.294e+03, 9.146e+03, 4.327e+03, 1.265e+03,
       9.851e+03, 1.502e+03, 5.227e+03, 2.836e+03, 7.701e+03, 2.109e+03,
       7.316e+03, 3.898e+03, 5.750e+03, 3.037e+03, 6.882e+03, 6.932e+03,
       7.344e+03, 7.953e+03, 4.000e+00, 4.054e+03, 1.969e+03, 1.881e+03,
       9.792e+03, 8.257e+03, 3.658e+03, 3.320e+02, 1.119e+03, 7.780e+02,
       2.963e+03, 7.406e+03, 6.625e+03, 4.008e+03, 7.540e+03, 4.380e+03,
       5.764e+03, 9.526e+03, 7.824e+03, 4.912e+03, 6.018e+03, 7.214e+03,
       1.100e+01, 5.255e+03, 7.644e+03, 9.700e+01, 7.675e+03, 5.907e+03,
       5.610e+02, 1.521e+03, 4.482e+03, 7.982e+03, 2.390e+03, 5.391e+03,
       8.736e+03, 3.219e+03, 3.654e+03, 8.851e+03, 7.608e+03, 2.000e+03,
       5.718e+03, 8.419e+03, 9.762e+03, 1.786e+03, 

In [12]:
print(f"Number of clusters: {len(kcentroid)}")

Number of clusters: 200
